# Recommendation System Using Cosine Similarity

The objective of this project is to build an anime recommendation system using cosine similarity. The system recommends similar anime based on genres, ratings, and other features.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,recall_score,f1_score,precision_score
from sklearn.model_selection import train_test_split

In [4]:
df=pd.read_csv('anime.csv')
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


# Data Preprocessing

In this step, the dataset is loaded and cleaned. Missing values are handled and the structure of the dataset is analyzed for further processing.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [7]:
df.shape

(12294, 7)

In [8]:
df.columns

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')

In [9]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [10]:
df.dropna(inplace=True)

In [11]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(0)

# Feature Extraction

Important features such as genre, ratings, and members are selected. Categorical data is converted into numerical form using CountVectorizer.

In [13]:
features = df[['name', 'genre', 'rating', 'episodes', 'members']]
features.head()

,name,genre,rating,episodes,members
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural",9.37,1,200630
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",9.26,64,793665
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",9.25,51,114262
3,Steins;Gate,"Sci-Fi, Thriller",9.17,24,673572
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",9.16,51,151266


In [14]:
cv = CountVectorizer(stop_words='english')

genre_matrix = cv.fit_transform(df['genre'])

In [15]:
scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(df[['rating', 'members']])

# Cosine Similarity

Cosine similarity measures the similarity between anime based on feature vectors. Higher similarity scores indicate more related anime.

In [16]:
cosine_sim = cosine_similarity(genre_matrix)

In [17]:
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

# Recommendation System

A recommendation function is created to suggest similar anime based on cosine similarity scores.

In [18]:
def recommend_anime(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    anime_indices = [i[0] for i in sim_scores]

    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [19]:
recommend_anime("One Piece")

,name,genre,rating
231,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29
241,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27
896,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78
352,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.16
753,One Piece: Episode of Luffy - Hand Island no B...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.86
941,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.76
1171,One Piece Movie 9: Episode of Chopper Plus - F...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.65
1576,One Piece: Adventure of Nebulandia,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.50
1793,One Piece Movie 5: Norowareta Seiken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.44
1795,One Piece: Umi no Heso no Daibouken-hen,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.44


In [20]:
def recommend_with_threshold(title, threshold=0.5):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    filtered = [i for i in sim_scores if i[1] > threshold]

    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)

    anime_indices = [i[0] for i in filtered[1:11]]

    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [21]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Model Evaluation

The recommendation system is evaluated using Precision, Recall, and F1-score to measure its effectiveness.

In [22]:
# Select anime
anime_name = "Naruto"

# Get similarity scores
idx = indices[anime_name]

sim_scores = list(enumerate(cosine_sim[idx]))

# Sort scores
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# Top 10 recommendations
recommended_indices = [i[0] for i in sim_scores[1:11]]

# Create relevant anime based on genre similarity
target_genre = df.iloc[idx]['genre']

relevant_indices = df[df['genre'] == target_genre].index.tolist()

# True labels
y_true = [1 if i in relevant_indices else 0 for i in recommended_indices]

# Predicted labels
y_pred = [1 for i in recommended_indices]

# Metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred)

print("Precision Score :", precision)
print("Recall Score    :", recall)
print("F1 Score        :", f1)

Precision Score : 0.7
Recall Score    : 1.0
F1 Score        : 0.8235294117647058


# Interview Questions and Answers

## 1. What is a Recommendation System?

A recommendation system is a machine learning technique used to suggest relevant items to users based on their preferences, behavior, or similarity between items.

---

## 2. What is Collaborative Filtering?

Collaborative filtering is a recommendation technique that predicts user interests by analyzing preferences and interactions of similar users or items.

---

## 3. What is the difference between User-Based and Item-Based Collaborative Filtering?

### User-Based Collaborative Filtering
- Recommends items based on similar users.
- Finds users with similar interests.
- Suggests items liked by similar users.

### Item-Based Collaborative Filtering
- Recommends items similar to the current item.
- Finds relationships between items.
- More scalable than user-based filtering.

---

## 4. What is Content-Based Filtering?

Content-based filtering recommends items based on item features such as genre, category, or description.

---

## 5. What is Cosine Similarity?

Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them.

The cosine similarity value ranges from:
- 0 → No similarity
- 1 → Highly similar

---

## 6. Why is Cosine Similarity used in Recommendation Systems?

Cosine similarity is used because it:
- Works well with text data
- Measures similarity effectively
- Handles high-dimensional data efficiently

---

## 7. What is Feature Extraction?

Feature extraction is the process of selecting important attributes from data for machine learning models.

Example:
- Genre
- Rating
- Episodes
- Members

---

## 8. What is CountVectorizer?

CountVectorizer converts textual data into numerical vectors based on word frequency.

---

## 9. Why do we normalize data?

Normalization scales numerical values into a fixed range to improve model performance and avoid bias from large values.

---

## 10. What are Precision, Recall, and F1-Score?

### Precision
Measures how many recommended items are actually relevant.

### Recall
Measures how many relevant items were successfully recommended.

### F1-Score
Harmonic mean of precision and recall.

---

## 11. What is the role of similarity threshold in recommendation systems?

Threshold values control the minimum similarity score required for recommendations. Higher thresholds produce more accurate but fewer recommendations.

---

## 12. What are the advantages of Recommendation Systems?

- Personalized suggestions
- Improved user experience
- Increased engagement
- Better decision-making

---

## 13. What are the limitations of Recommendation Systems?

- Cold start problem
- Data sparsity
- Scalability issues
- Limited personalization with insufficient data

---

## 14. What is the Cold Start Problem?

The cold start problem occurs when the system lacks sufficient data about new users or items, making recommendations difficult.

---

## 15. What is Data Sparsity?

Data sparsity occurs when users interact with only a small subset of available items, resulting in sparse datasets.

---

## 16. What is a Hybrid Recommendation System?

A hybrid recommendation system combines multiple recommendation techniques such as collaborative filtering and content-based filtering.

---

## 17. Why is preprocessing important in machine learning?

Preprocessing improves data quality by:
- handling missing values
- removing duplicates
- converting categorical data
- scaling numerical values

---

## 18. What is overfitting in recommendation systems?

Overfitting occurs when a model performs well on training data but poorly on unseen data.

---

## 19. What is the purpose of train-test split?

Train-test split separates data into training and testing sets to evaluate model performance on unseen data.

---

## 20. What libraries were used in this project?

Libraries used:
- Pandas
- NumPy
- Scikit-learn

---

## 21. Why is Pandas used?

Pandas is used for:
- data manipulation
- cleaning
- analysis
- handling DataFrames

---

## 22. Why is NumPy used?

NumPy is used for numerical computations and array operations.

---

## 23. What is Scikit-learn?

Scikit-learn is a Python machine learning library used for:
- preprocessing
- model building
- evaluation
- similarity computation

---

## 24. What is a vector in machine learning?

A vector is a numerical representation of data used for mathematical computations.

---

## 25. What improvements can be made to this recommendation system?

Possible improvements:
- Use hybrid recommendation systems
- Include user ratings
- Apply deep learning techniques
- Improve feature engineering
- Use collaborative filtering